# Content-Based and Hybrid Recommenders — Exercises

Companion to the note [Content-Based and Hybrid Recommenders](Content-Based%20and%20Hybrid%20Recommenders.md).

Practise content-based recommendation (item features, TF-IDF, user profiles, per-user classifiers), hybrid strategies (weighted, switching), cold-start reasoning, and the over-specialisation problem with a diversity re-ranker (MMR). Everything uses a tiny movie catalogue with genre tags and short synthetic descriptions.

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×5 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import expit
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import BernoulliNB
rng = np.random.default_rng(0)

genres = ['Action', 'Comedy', 'Drama', 'SciFi', 'Romance']
# 8 movies x 5 genre flags
G = np.array([[1, 0, 0, 1, 0],   # 0 Action+SciFi
              [1, 0, 0, 0, 0],   # 1 Action
              [0, 1, 0, 0, 1],   # 2 Comedy+Romance
              [0, 0, 1, 0, 1],   # 3 Drama+Romance
              [0, 0, 1, 1, 0],   # 4 Drama+SciFi
              [0, 1, 0, 0, 0],   # 5 Comedy
              [1, 1, 0, 0, 0],   # 6 Action+Comedy
              [0, 0, 0, 1, 0]],  # 7 SciFi
             dtype=float)
descriptions = ["space marines fight alien robots in deep space",
                "a lone hero fights a crime gang",
                "two strangers fall in love in paris comedy",
                "a family drama about love and loss",
                "a scientist drama on a lonely space station",
                "a silly comedy about a talking dog",
                "action comedy with a clumsy hero cop",
                "robots and aliens explore deep space"]
# Ratings of 4 users on the 8 movies (0 = unrated)
Ru = np.array([[5, 4, 1, 0, 0, 0, 0, 0],
               [0, 0, 5, 4, 0, 2, 0, 0],
               [0, 5, 0, 0, 0, 4, 0, 0],
               [4, 0, 0, 0, 5, 0, 0, 0]], dtype=float)

## Part A · Concepts

### Exercise 1 · Strengths and weaknesses
*🧠 Concept · ★☆☆*

The note lists pros (+) and cons (−) of content-based recommenders. For each, give a concrete example:

1. No cold start for new **items**. 2. Explainable. 3. Over-specialisation (filter bubble). 4. Needs good features.
5. New **users** are still cold.

Why can a content-based system never recommend something "surprisingly good" (serendipity) the way CF can?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Content-based uses only *this* user's history plus item features. CF uses *other users'* behaviour.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. A movie released today has genre tags and a plot, so it can be scored immediately, with no ratings needed.
2. "Recommended because you liked *Alien* (SciFi, space)."
3. A user who rated two SciFi films gets only SciFi forever.
4. Two films both tagged "Drama" can be completely different. Tags miss style, quality and tone; text/image embeddings help ([[Text Representations]]).
5. A user with no history has no profile to compare against.

**Serendipity:** content-based can only recommend items close to the profile in feature space. CF can discover that people who love
SciFi also tend to love a particular jazz documentary, a link no feature encodes. This is why hybrids are popular.

</details>

### Exercise 2 · Pick the hybrid
*🧠 Concept · ★☆☆*

Match each scenario to a hybrid type from the note (weighted, switching, feature augmentation, model-level):

(a) A ranker (GBDT) gets the MF score as one of its input features. (b) New items (<10 ratings) are scored by content, others by CF.
(c) Final score $= 0.7\cdot\text{CF} + 0.3\cdot\text{content}$. (d) LightFM: a user/item embedding is the sum of embeddings of its features (including its ID).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Is the combination happening at the output (scores), by routing, as an input to another model, or inside a single model?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Feature augmentation**. (b) **Switching**. (c) **Weighted**. (d) **Model-level** (side features inside one model, like factorization
machines or two-tower models with side features, see [[Deep Learning Recommenders]]).

</details>

### Exercise 3 · Cold start: new user vs new item
*🧠 Concept · ★★☆*

For each strategy from the note, say whether it addresses the **new-user** or the **new-item** problem (or both), and its main downside:
onboarding questionnaire, popularity by segment, content/meta features, bandit exploration, cross-domain transfer.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ask: whose history is missing, and what information is available instead?

</details>

<details>
<summary><b>✅ Solution</b></summary>

| Strategy | Helps | Downside |
|---|---|---|
| Onboarding questionnaire | new user | friction, people quit or answer carelessly |
| Popularity by segment (country, age, device) | new user | coarse, not personal |
| Content/meta features | mainly new item (also new users with demographics) | needs good features |
| Bandit exploration ([[Q-Learning and Policy Gradients]]) | both | short-term cost of showing worse items |
| Cross-domain transfer | new user (e.g. books → movies) | needs user overlap, domain shift |

</details>

### Exercise 4 · When knowledge-based wins
*🧠 Concept · ★☆☆*

Why do car or real-estate sites use **knowledge-based / constraint-based** recommendation rather than CF?
Give two properties of these domains that break CF and explain how constraints (budget, size, location) solve them.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

How often does someone buy a house? Do their past purchases predict the next one?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Rare purchases:** users buy a house or car a few times in their lives, so there is almost no interaction history per user (permanent cold start).
2. **Hard requirements that change:** a family needs 4 bedrooms under €400k *now*. Past taste is a poor predictor, and violating a constraint
   makes a recommendation useless, however "similar" it is.

Constraint-based systems filter the catalogue by explicit requirements and rank what remains (by utility or similarity), so they need no history.

</details>

## Part B · By hand

### Exercise 5 · A user profile and cosine scores
*✍️ By hand · ★☆☆*

User 0 in `Ru` liked movies 0 (rating 5) and 1 (rating 4) and disliked movie 2 (rating 1). Build the profile as the
**rating-weighted average of liked items** (rating ≥ 4) in genre space, then compute the cosine score of unseen movies 4, 6 and 7.
Which of the three would you recommend?

In [ ]:
scores_467 = None   # [score4, score6, score7]
best_unseen = None

_prof = (5 * G[0] + 4 * G[1]) / 9
_ref = cosine_similarity(_prof[None], G[[4, 6, 7]])[0]
check('cosine scores', scores_467, _ref, tol=1e-3)
check('best of the three', best_unseen, [4, 6, 7][int(np.argmax(_ref))])

<details>
<summary><b>💡 Hint 1</b></summary>

Profile $= (5\cdot(1,0,0,1,0) + 4\cdot(1,0,0,0,0))/9 = (1, 0, 0, 5/9, 0)$, with norm $\sqrt{106}/9$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Movie 6 = (1,1,0,0,0), movie 7 = (0,0,0,1,0), movie 4 = (0,0,1,1,0).

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\lVert\text{profile}\rVert = \sqrt{1 + 25/81} = \sqrt{106}/9$.

- Movie 4: dot $=5/9$, $\cos = \frac{5/9}{(\sqrt{106}/9)\sqrt2} = 5/\sqrt{212} \approx 0.343$.
- Movie 6: dot $=1$, $\cos = 9/\sqrt{212} \approx 0.618$.
- Movie 7: dot $=5/9$, $\cos = 5/\sqrt{106} \approx 0.486$.

Recommend **movie 6** (Action+Comedy), even though the user disliked a comedy. The positive-only profile ignores negative evidence (next exercise).

```python
scores_467 = [5 / np.sqrt(212), 9 / np.sqrt(212), 5 / np.sqrt(106)]
best_unseen = 6
```

</details>

### Exercise 6 · Using negative feedback: a centred profile
*✍️ By hand · ★★☆*

Instead, build the profile from **all** of user 0's ratings, weighted by the rating minus the user's mean
(a Rocchio-style profile): $\mathbf u = \sum_i (r_{ui}-\bar r_u)\,\mathbf x_i$. Compute $\mathbf u$ and the dot-product scores of
movies 4, 6, 7. What changed compared with Exercise 5?

In [ ]:
profile_c = None  # 5 numbers
scores_c = None   # [s4, s6, s7]

_r = Ru[0]; _m = _r > 0; _w = (_r - _r[_m].mean()) * _m
check('centred profile', profile_c, _w @ G, tol=1e-3)
check('scores', scores_c, (_w @ G) @ G[[4, 6, 7]].T, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\bar r_0 = 10/3$; the weights are $5/3$, $2/3$, $-7/3$ for movies 0, 1, 2.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathbf u = \tfrac53(1,0,0,1,0) + \tfrac23(1,0,0,0,0) - \tfrac73(0,1,0,0,1) = (7/3,\ -7/3,\ 0,\ 5/3,\ -7/3)$.

Scores: movie 4 $= 5/3 \approx 1.667$, movie 6 $= 7/3 - 7/3 = 0$, movie 7 $= 5/3$.
The disliked comedy now cancels the Action signal for movie 6, and the SciFi movies win. Using negatives makes profiles sharper.

```python
profile_c = [7/3, -7/3, 0, 5/3, -7/3]
scores_c = [5/3, 0, 5/3]
```

</details>

### Exercise 7 · TF-IDF by hand
*✍️ By hand · ★☆☆*

Corpus of three item descriptions: $d_1$ = "space war robots", $d_2$ = "space love story", $d_3$ = "love love comedy".
Using raw term frequency and $\mathrm{idf}(t) = \ln(N/\mathrm{df}(t))$, compute $\text{tf-idf}(\text{love}, d_3)$ and
$\text{tf-idf}(\text{comedy}, d_3)$. Why does "comedy" get a higher weight per occurrence than "love"?

In [ ]:
tfidf_love_d3 = None
tfidf_comedy_d3 = None

check('love in d3', tfidf_love_d3, 2 * np.log(3 / 2))
check('comedy in d3', tfidf_comedy_d3, 1 * np.log(3 / 1))

<details>
<summary><b>💡 Hint</b></summary>

$N=3$; 'love' appears in 2 documents, 'comedy' in 1.

</details>

<details>
<summary><b>✅ Solution</b></summary>

love: $\mathrm{tf}=2$, $\mathrm{idf}=\ln(3/2)\approx0.405$, so tf-idf $\approx 0.811$.
comedy: $\mathrm{tf}=1$, $\mathrm{idf}=\ln 3\approx1.099$, so tf-idf $\approx 1.099$.
Rare terms are more discriminative: "comedy" singles out $d_3$, while "love" is shared with $d_2$.
(sklearn uses a smoothed $\ln\frac{1+N}{1+\mathrm{df}}+1$ and L2-normalises rows; see Exercise 10.)

```python
tfidf_love_d3 = 2 * np.log(1.5)
tfidf_comedy_d3 = np.log(3)
```

</details>

### Exercise 8 · A per-user logistic profile
*✍️ By hand · ★☆☆*

A per-user [[Logistic Regression]] on genre flags has learned $w = (1.2,\ -0.5,\ 0,\ 0.8,\ -1.0)$ (Action, Comedy, Drama, SciFi, Romance)
and bias $b=-0.5$. Compute $P(\text{like})=\sigma(w^\top x + b)$ for movie 0 (Action+SciFi) and movie 6 (Action+Comedy).

In [ ]:
p_like_0 = None
p_like_6 = None

_w = np.array([1.2, -.5, 0, .8, -1.]); check('movie 0', p_like_0, expit(G[0] @ _w - .5), tol=1e-3)
check('movie 6', p_like_6, expit(G[6] @ _w - .5), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Movie 0: $z = 1.2 + 0.8 - 0.5$. Movie 6: $z = 1.2 - 0.5 - 0.5$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Movie 0: $z=1.5$, $\sigma(1.5)\approx 0.818$. Movie 6: $z=0.2$, $\sigma(0.2)\approx 0.550$.
Unlike a fixed similarity, the classifier learns *negative* genre weights and a calibrated probability. It needs enough ratings per user, though.

```python
p_like_0 = expit(1.5)
p_like_6 = expit(0.2)
```

</details>

### Exercise 9 · Weighted hybrid and score scales
*✍️ By hand · ★★☆*

For three candidate items a CF model gives scores $(0.2,\ 0.9,\ 0.5)$ and a content model gives $(10,\ 4,\ 7)$.

1. Blend raw scores: $0.6\cdot\text{CF} + 0.4\cdot\text{content}$. Which item wins?
2. First min-max normalise each model's scores to $[0,1]$ over the three items, then blend with the same weights. Which item wins now?

Store the normalised blended scores in `blend_norm`.

In [ ]:
blend_norm = None   # 3 numbers
winner_raw = None
winner_norm = None

_cf = np.array([.2, .9, .5]); _ct = np.array([10, 4, 7.])
_mm = lambda x: (x - x.min()) / (x.max() - x.min())
check('normalised blend', blend_norm, .6 * _mm(_cf) + .4 * _mm(_ct), tol=1e-3)
check('raw winner', winner_raw, int(np.argmax(.6 * _cf + .4 * _ct)))
check('normalised winner', winner_norm, int(np.argmax(.6 * _mm(_cf) + .4 * _mm(_ct))))

<details>
<summary><b>💡 Hint</b></summary>

CF normalised: $(0, 1, 0.3/0.7)$. Content normalised: $(1, 0, 0.5)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Raw: $(0.12+4.0,\ 0.54+1.6,\ 0.3+2.8) = (4.12, 2.14, 3.10)$, so **item 0** wins. The content scores have a 10× larger scale and dominate,
   whatever the weights say.
2. Normalised: CF $\to(0, 1, 0.4286)$, content $\to(1, 0, 0.5)$; blend $=(0.4,\ 0.6,\ 0.4571)$, so **item 1** wins.

Always calibrate or normalise scores before blending (min-max, z-score, rank, or learn the blend in a model).

```python
blend_norm = [0.4, 0.6, 0.6 * 3 / 7 + 0.2]
winner_raw = 0
winner_norm = 1
```

</details>

## Part C · Code from scratch

### Exercise 10 · TF-IDF like sklearn
*💻 Code · ★★☆*

Implement `tfidf(docs)` that returns `(vocab, X)` where `vocab` is the alphabetically sorted list of words with ≥2 characters
and `X` is the documents × vocab matrix with sklearn's defaults:
$\mathrm{tf}$ = raw count, $\mathrm{idf}(t) = \ln\frac{1+N}{1+\mathrm{df}(t)} + 1$, then each row L2-normalised.
Apply it to `descriptions`.

In [ ]:
def tfidf(docs):
    # TODO: tokenise with str.split(), keep words with len >= 2
    return None

tf_out = tfidf(descriptions)

_v = TfidfVectorizer(); _X = _v.fit_transform(descriptions).toarray()
if tf_out is not None:
    check('vocabulary', list(tf_out[0]) == list(_v.get_feature_names_out()), True)
    check('tf-idf matrix', tf_out[1], _X)
else:
    check('tf-idf', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

Build `vocab = sorted({w for d in docs for w in d.split() if len(w) >= 2})`, count into a matrix, `df = (C > 0).sum(0)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The `+1` terms smooth the idf (as if one extra document contained every word) and keep terms that appear everywhere from getting weight 0.
L2 normalisation makes the dot product between rows equal to their cosine similarity.

```python
def tfidf(docs):
    toks = [[w for w in d.split() if len(w) >= 2] for d in docs]
    vocab = sorted({w for t in toks for w in t})
    idx = {w: j for j, w in enumerate(vocab)}
    C = np.zeros((len(docs), len(vocab)))
    for r, t in enumerate(toks):
        for w in t:
            C[r, idx[w]] += 1
    N = len(docs)
    idf = np.log((1 + N) / (1 + (C > 0).sum(0))) + 1
    X = C * idf
    return vocab, X / np.linalg.norm(X, axis=1, keepdims=True)

tf_out = tfidf(descriptions)
```

</details>

### Exercise 11 · Content-based top-K
*💻 Code · ★☆☆*

Implement `recommend_content(R, F, u, K)`: build user `u`'s profile as the rating-weighted average of the item feature vectors
they rated ≥ 4, score **unrated** items by cosine similarity with the profile, and return the top-K item indices (ties by lower index).
Run it for every user of `Ru` with the text features `F_text = tf_out[1]` and K=2.

In [ ]:
def recommend_content(R, F, u, K):
    # TODO
    return None

F_text = None if tf_out is None else tf_out[1]
recs_content = None   # list of 4 lists

_F = TfidfVectorizer().fit_transform(descriptions).toarray()
def _ref(u, K=2):
    liked = Ru[u] >= 4
    prof = (Ru[u, liked] @ _F[liked]) / Ru[u, liked].sum()
    s = cosine_similarity(prof[None], _F)[0]
    return sorted([i for i in range(8) if Ru[u, i] == 0], key=lambda i: (-s[i], i))[:K]
check('top-2 lists', recs_content, [_ref(u) for u in range(4)])

<details>
<summary><b>💡 Hint</b></summary>

`liked = R[u] >= 4`; `prof = R[u, liked] @ F[liked] / R[u, liked].sum()`; then cosine against all rows, mask rated items.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Result: `[[7, 4], [0, 7], [6, 3], [7, 2]]`. User 0 (liked space marines and a crime hero) gets movie 7 ("robots and aliens explore deep space")
and movie 4 ("a scientist drama on a lonely space station"); the word *space* drives both. Text overlap gives explanations for free, but it is brittle:
user 1 (romance/comedy fan) is recommended the space films 0 and 7 only through the stop words "in" and "and". Stop-word removal and better
embeddings ([[Text Representations]]) matter.

```python
def recommend_content(R, F, u, K):
    liked = R[u] >= 4
    prof = R[u, liked] @ F[liked] / R[u, liked].sum()
    s = F @ prof / (np.linalg.norm(F, axis=1) * np.linalg.norm(prof) + 1e-12)
    cand = np.flatnonzero(R[u] == 0)
    return [int(i) for i in cand[np.argsort(-s[cand], kind='stable')[:K]]]

F_text = tf_out[1]
recs_content = [recommend_content(Ru, F_text, u, 2) for u in range(4)]
print(recs_content)
```

</details>

### Exercise 12 · A per-user Naive Bayes classifier
*💻 Code · ★★☆*

A user labelled 40 catalogue items as liked (1) / not liked (0); items have 6 binary tags (`X_nb`, `y_nb` below).
Implement **Bernoulli [[Naive Bayes]]** from scratch with Laplace smoothing
$P(x_j=1\mid c) = \frac{n_{cj}+1}{n_c+2}$, and return `predict_proba` for new items. Compare with `sklearn.naive_bayes.BernoulliNB(alpha=1)`.

In [ ]:
_g = np.random.default_rng(11)
X_nb = (_g.random((40, 6)) < 0.4).astype(float)
y_nb = (((X_nb[:, 0] + X_nb[:, 3]) >= 1) ^ (_g.random(40) < 0.1)).astype(int)
X_new = (_g.random((5, 6)) < 0.4).astype(float)

def nb_fit_predict(X, y, X_new):
    # TODO: priors, smoothed feature probabilities, log-likelihoods, normalise -> (n_new, 2) probabilities
    return None

proba_nb = nb_fit_predict(X_nb, y_nb, X_new)

check('predict_proba', proba_nb, BernoulliNB(alpha=1.0).fit(X_nb, y_nb).predict_proba(X_new))

<details>
<summary><b>💡 Hint 1</b></summary>

$\log P(c\mid x) = \log P(c) + \sum_j [x_j\log\theta_{cj} + (1-x_j)\log(1-\theta_{cj})] + \text{const}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Normalise with log-sum-exp: subtract the row max before exponentiating.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Bernoulli NB also uses the *absence* of a tag as evidence. It trains in one pass, works with very few ratings, and its
per-tag log-odds are a readable explanation of the user's taste. With strongly correlated tags, the independence assumption makes probabilities overconfident.

```python
_g = np.random.default_rng(11)
X_nb = (_g.random((40, 6)) < 0.4).astype(float)
y_nb = (((X_nb[:, 0] + X_nb[:, 3]) >= 1) ^ (_g.random(40) < 0.1)).astype(int)
X_new = (_g.random((5, 6)) < 0.4).astype(float)

def nb_fit_predict(X, y, X_new):
    classes = np.unique(y)
    logp = []
    for c in classes:
        Xc = X[y == c]
        theta = (Xc.sum(0) + 1) / (len(Xc) + 2)
        prior = np.log(len(Xc) / len(X))
        logp.append(prior + X_new @ np.log(theta) + (1 - X_new) @ np.log(1 - theta))
    L = np.array(logp).T
    L -= L.max(1, keepdims=True)
    P = np.exp(L)
    return P / P.sum(1, keepdims=True)

proba_nb = nb_fit_predict(X_nb, y_nb, X_new)
```

</details>

### Exercise 13 · A switching hybrid
*💻 Code · ★☆☆*

Implement `switching_scores(cf, content, n_ratings, threshold)`: for each item use the **content** score if it has fewer than
`threshold` ratings, else the **CF** score. Then return the top-3 item indices for the example below.
Why might the two score types still need calibration even in a switching hybrid?

In [ ]:
cf_s = np.array([0.9, 0.1, 0.7, 0.4, 0.8, 0.3])
ct_s = np.array([0.2, 0.95, 0.5, 0.6, 0.1, 0.85])
n_rat = np.array([120, 2, 40, 0, 300, 5])

def switching_scores(cf, content, n_ratings, threshold=10):
    return None

top3_switch = None

_s = [ct_s[i] if n_rat[i] < 10 else cf_s[i] for i in range(6)]
check('scores', switching_scores(cf_s, ct_s, n_rat, 10), _s)
check('top-3', top3_switch, sorted(range(6), key=lambda i: -_s[i])[:3])

<details>
<summary><b>💡 Hint</b></summary>

`np.where(n_ratings < threshold, content, cf)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Scores $= (0.9, 0.95, 0.7, 0.6, 0.8, 0.85)$, so the top-3 are items 1, 0 and 5. Two of the three are cold items scored by content.
If the content model is systematically more "confident" (higher scores) than CF, cold items will flood the list.
The scores live on different scales, so calibrate them (e.g. to probabilities) before mixing them in one ranking.

```python
cf_s = np.array([0.9, 0.1, 0.7, 0.4, 0.8, 0.3])
ct_s = np.array([0.2, 0.95, 0.5, 0.6, 0.1, 0.85])
n_rat = np.array([120, 2, 40, 0, 300, 5])

def switching_scores(cf, content, n_ratings, threshold=10):
    return np.where(n_ratings < threshold, content, cf)

top3_switch = [int(i) for i in np.argsort(-switching_scores(cf_s, ct_s, n_rat, 10), kind='stable')[:3]]
print(top3_switch)
```

</details>

### Exercise 14 · Tuning a weighted hybrid
*💻 Code · ★★☆*

Simulated setting: 60 users × 50 items with a true affinity matrix `T`. Each user's held-out item is their highest-affinity one.
Two models see the truth through **independent** noise: `cf = T + noise`, `ct = T + noise`.

Implement `hit_rate(S, pos, K)` (fraction of users whose held-out item is in their top-K by score `S`) and evaluate the blend
$\alpha\cdot\text{cf} + (1-\alpha)\cdot\text{ct}$ for $\alpha\in\{0, 0.1, \dots, 1\}$ with K=10. Store the hit rates in `hr_by_alpha`
(11 numbers) and the best α in `best_alpha`. Why is the best α strictly between 0 and 1?

In [ ]:
_g = np.random.default_rng(3)
T = _g.normal(size=(60, 50)); pos_items = np.argmax(T, 1)
cf_scores = T + _g.normal(0, 1.2, T.shape); ct_scores = T + _g.normal(0, 1.2, T.shape)
alphas = np.round(np.linspace(0, 1, 11), 1)

def hit_rate(S, pos, K=10):
    return None

hr_by_alpha = None
best_alpha = None

def _hr(S, K=10):
    return np.mean([pos_items[u] in set(np.argsort(-S[u])[:K]) for u in range(len(S))])
_ref = [_hr(a * cf_scores + (1 - a) * ct_scores) for a in alphas]
check('hit rates', hr_by_alpha, _ref)
check('best alpha', best_alpha, alphas[int(np.argmax(_ref))])

<details>
<summary><b>💡 Hint</b></summary>

Top-K per row: `np.argsort(-S, axis=1)[:, :K]`; then check membership of `pos[u]` row by row.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The hit rate is ≈0.72 for content alone and ≈0.77 for CF alone, and peaks at ≈0.88 for $\alpha=0.4$. Averaging two estimates with **independent** errors reduces
the noise variance (like an ensemble), so the blend beats both. With perfectly correlated errors blending would not help.
In practice tune α on a validation split, never on the test set ([[Evaluating Recommenders]]).

```python
_g = np.random.default_rng(3)
T = _g.normal(size=(60, 50)); pos_items = np.argmax(T, 1)
cf_scores = T + _g.normal(0, 1.2, T.shape); ct_scores = T + _g.normal(0, 1.2, T.shape)
alphas = np.round(np.linspace(0, 1, 11), 1)

def hit_rate(S, pos, K=10):
    top = np.argsort(-S, axis=1)[:, :K]
    return float(np.mean([pos[u] in top[u] for u in range(len(S))]))

hr_by_alpha = [hit_rate(a * cf_scores + (1 - a) * ct_scores, pos_items) for a in alphas]
best_alpha = alphas[int(np.argmax(hr_by_alpha))]
print(np.round(hr_by_alpha, 3), best_alpha)
```

</details>

### Exercise 15 · Intra-list similarity
*💻 Code · ★★☆*

**Intra-list similarity** (ILS) of a list $L$ is the mean pairwise similarity $\frac{2}{|L|(|L|-1)}\sum_{a<b}\mathrm{sim}(a,b)$
(lower = more diverse). Implement `ils(L, S)` for an item–item similarity matrix `S`.

On the catalogue below (30 items in 3 genre clusters, relevance highest for cluster A), compute the ILS of the top-5 by relevance.

In [ ]:
_g = np.random.default_rng(4)
F_cl = np.vstack([np.eye(3)[c] * 3 + _g.normal(0, .5, 3) for c in np.repeat(range(3), 10)])
rel = np.r_[_g.uniform(.7, 1, 10), _g.uniform(.4, .8, 10), _g.uniform(.3, .7, 10)]
S_cl = cosine_similarity(F_cl)

def ils(L, S):
    return None

top5_rel = list(np.argsort(-rel)[:5])
ils_top5 = None

import itertools
check('ILS of top-5', ils_top5, np.mean([S_cl[a, b] for a, b in itertools.combinations(top5_rel, 2)]))

<details>
<summary><b>💡 Hint</b></summary>

Loop over pairs with `itertools.combinations`, or take the mean of the upper triangle of `S[np.ix_(L, L)]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

All five most relevant items come from cluster A, so the ILS is ≈0.93, a perfect filter bubble. This is the over-specialisation the note warns about.

```python
_g = np.random.default_rng(4)
F_cl = np.vstack([np.eye(3)[c] * 3 + _g.normal(0, .5, 3) for c in np.repeat(range(3), 10)])
rel = np.r_[_g.uniform(.7, 1, 10), _g.uniform(.4, .8, 10), _g.uniform(.3, .7, 10)]
S_cl = cosine_similarity(F_cl)

def ils(L, S):
    L = list(L)
    sub = S[np.ix_(L, L)]
    iu = np.triu_indices(len(L), 1)
    return float(sub[iu].mean())

top5_rel = list(np.argsort(-rel)[:5])
ils_top5 = ils(top5_rel, S_cl)
print(round(ils_top5, 3))
```

</details>

### Exercise 16 · Diversity re-ranking with MMR
*💻 Code · ★★★*

**Maximal Marginal Relevance** builds the list greedily:
$$i^\star = \arg\max_{i\notin L}\ \lambda\,\mathrm{rel}(i) - (1-\lambda)\max_{j\in L}\mathrm{sim}(i,j)$$
(the max term is 0 while $L$ is empty). Implement `mmr(rel, S, K, lam)` and compare, for K=5 and $\lambda=0.5$, the ILS and the total
relevance of the MMR list against the pure top-5. Store the MMR list in `mmr5`.

In [ ]:
def mmr(rel, S, K, lam):
    # TODO: greedy selection
    return None

mmr5 = mmr(rel, S_cl, 5, 0.5)

if mmr5 is not None:
    check('lambda = 1 reproduces the top-K by relevance', list(mmr(rel, S_cl, 5, 1.0)), list(np.argsort(-rel)[:5]))
    check('first pick is the most relevant item', mmr5[0], int(np.argmax(rel)))
    import itertools
    _ils = lambda L: np.mean([S_cl[a, b] for a, b in itertools.combinations(L, 2)])
    check('MMR list is far more diverse', _ils(mmr5) < _ils(list(np.argsort(-rel)[:5])) - 0.5, True)
    check('MMR covers all 3 clusters', len({int(i) // 10 for i in mmr5}), 3)
else:
    check('MMR', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Keep `selected = []` and `candidates = list(range(n))`; each step pick the candidate maximising the MMR objective and move it.

</details>

<details>
<summary><b>✅ Solution</b></summary>

MMR picks the most relevant item from cluster A, then items from clusters B and C, and comes back to A only when its relevance outweighs the
redundancy penalty. ILS drops from ≈0.93 to ≈0.20, while total relevance drops only from ≈4.54 to ≈4.19. That is a small accuracy cost for a much more
varied list. In the multi-stage architecture this belongs in **re-ranking** ([[Recommender Systems Overview]]).

```python
def mmr(rel, S, K, lam):
    selected, cand = [], list(range(len(rel)))
    while len(selected) < K:
        def obj(i):
            red = max(S[i, selected]) if selected else 0.0
            return lam * rel[i] - (1 - lam) * red
        best = max(cand, key=obj)
        selected.append(int(best)); cand.remove(best)
    return selected

mmr5 = mmr(rel, S_cl, 5, 0.5)
print('MMR:', mmr5, 'ILS', round(ils(mmr5, S_cl), 3), 'rel', round(rel[mmr5].sum(), 3))
print('top:', top5_rel, 'ILS', round(ils(top5_rel, S_cl), 3), 'rel', round(rel[top5_rel].sum(), 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Content-Based and Hybrid Recommenders](Content-Based%20and%20Hybrid%20Recommenders.md).*